# 7 - PACE starter (Arabian Sea)

First pass at running the gap-fill U-Net on **PACE** instead of the CMEMS `IO.zarr`. Same idea, adapted
to PACE's format:

- PACE has **only `chlor_a`** (no SST/wind, and no L4 gap-free truth), so we evaluate **self-supervised**
  with the fake-cloud metric and derive the land / cloud flags ourselves.
- The Arabian Sea PACE crop is ~256x376 and **one spatial chunk per day**, so it fits whole - no tiling
  needed yet (tiling / stitching is only for the global extent later).
- prev/next-day channels are built from the **masked** CHL (Eli's leak fix).

This is a scaffold: it needs the Hub (earthaccess + AWS us-west-2) to read PACE, and the open decisions
at the bottom still need answers. Marked NOTE/TODO where it needs validation.

## Setup

**Run this on AWS us-west-2** (e.g. CryoCloud): the PACE `inregion` store only opens from there.

The two cells below (a) install the `mindthegap` package and (b) upgrade `icechunk` to a version that has
`http_storage` (the Hub default is too old). **Restart the kernel after they finish**, then run the rest
top to bottom. `earthaccess.login()` in the next cell prompts for NASA Earthdata credentials.

In [1]:
%pip install --force-reinstall --no-cache-dir "git+https://github.com/SAFS-Varanasi-Internship/mindthegap.git@main"


In [2]:
!pip install -qU icechunk

import icechunk as ic
print(ic.__version__, hasattr(ic, "http_storage"))


In [3]:
import earthaccess
import icechunk as ic
import xarray as xr
import numpy as np, pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import mindthegap as mtg

def create_ds(product="PACE_OCI_L3M_CHL", group="daily/0p1deg/chunks_512"):
    """Open a PACE Icechunk store group as xarray (Eli's helper). Needs AWS us-west-2 + earthaccess login."""
    url = f"https://data.source.coop/fish-pace/pace-oci/inregion/{product}"
    storage = ic.http_storage(url)
    auth = earthaccess.login()
    creds = auth.get_s3_credentials(daac="OBDAAC")
    vc = ic.credentials.containers_credentials({
        "s3://ob-cumulus-prod-public/": ic.credentials.s3_credentials(
            access_key_id=creds["accessKeyId"], secret_access_key=creds["secretAccessKey"],
            session_token=creds["sessionToken"])})
    store = ic.Repository.open(storage, authorize_virtual_chunk_access=vc).readonly_session("main").store
    return xr.open_zarr(store, consolidated=False, group=group, chunks={})

## 1. Load PACE data

Arabian Sea `chlor_a` as 3-day composites (fuller coverage than daily). Cached to a local zarr so the slow
first read happens only once. Switch the group to `8Day/...`, or drop the `resample`, for other cadences.

In [4]:
import os
CACHE = "pace_arabsea_3day.zarr"
if os.path.exists(CACHE):
    zarr_ds = xr.open_zarr(CACHE)
else:
    ds = create_ds("PACE_OCI_L3M_CHL", "daily/0p1deg/chunks_512")
    zarr_ds = ds.sel(lat=slice(31, 5), lon=slice(42, 80)).drop_vars("palette")
    zarr_ds = mtg.crop_to_multiple(zarr_ds, multiple=8)
    zarr_ds = zarr_ds.resample(time="3D").mean().chunk({"time": 1})   # 3-day composites (skips NaN -> fewer gaps)
    zarr_ds.to_zarr(CACHE, mode="w")                                  # one-time; later runs are instant
print("dims:", dict(zarr_ds.sizes))
chl = np.log(zarr_ds["chlor_a"].where(zarr_ds["chlor_a"] > 0).load().values).astype("float32")
times = zarr_ds.time.values
print("chl cube:", chl.shape, " observed fraction:", round(float(np.mean(np.isfinite(chl))), 3))


## 2. Build the self-supervised channels

PACE has no gap-free truth, so we train and evaluate with **fake clouds**: hide observed pixels, predict
them, score only there. Fake clouds are small cloud-shaped blobs; the big orbital swaths are filtered out
by size (`max_blob_frac`). The prev/next-day channels are built from the *masked* CHL, so nothing leaks.

In [5]:
from scipy import ndimage

def build_pace_channels(chl_log, times, train_slice, cloud_len=3, max_blob_frac=0.01,
                        standardize_chl=True):
    """Fake clouds now mimic CLOUDS, not swaths: from a borrowed real-gap footprint keep only the
    small blobby connected components (area <= max_blob_frac of the ocean) and drop the big elongated
    swaths, so we stop masking half the ocean. Persistent across cloud_len steps; prev/next from the
    masked CHL (no leak). Lower max_blob_frac -> smaller clouds / less masking."""
    T, H, W = chl_log.shape
    gap = np.isnan(chl_log)
    land = gap.all(axis=0); land3 = np.broadcast_to(land, chl_log.shape)
    real_cloud = gap & ~land3
    blob_cap = max(1, int(max_blob_frac * int((~land).sum())))

    fake = np.zeros((T, H, W), bool)
    for s in range(0, T, cloud_len):
        e = min(s + cloud_len, T)
        foot = gap[(s + T // 2) % T] & ~land                 # borrowed real-gap footprint
        lab, n = ndimage.label(foot)                          # connected components
        if n:
            counts = np.bincount(lab.ravel())
            small = np.where(counts <= blob_cap)[0]
            small = small[small != 0]                         # drop background + big swaths
            foot = np.isin(lab, small)                        # keep only cloud-sized blobs
        fake[s:e] = foot[None] & ~gap[s:e] & ~land3[s:e]      # held fixed across the block

    masked = np.where(fake, np.nan, chl_log)
    prev = np.roll(masked, 1, axis=0);  prev[0] = np.nan
    nextd = np.roll(masked, -1, axis=0); nextd[-1] = np.nan
    doy = pd.to_datetime(times).dayofyear.to_numpy().astype("float32")
    ang = 2 * np.pi * doy / 365.0
    sin_t = np.broadcast_to(np.sin(ang)[:, None, None], chl_log.shape).astype("float32").copy()
    cos_t = np.broadcast_to(np.cos(ang)[:, None, None], chl_log.shape).astype("float32").copy()
    valid = (~np.isnan(masked)) & ~land3

    ch = {"sin_time": sin_t, "cos_time": cos_t,
          "masked_CHL": masked, "prev_day_CHL": prev, "next_day-CHL": nextd,
          "land_flag": land3.astype("float32"), "real_cloud_flag": real_cloud.astype("float32"),
          "valid_CHL_flag": valid.astype("float32"), "fake_cloud_flag": fake.astype("float32")}
    stats = {}
    for k in ["sin_time", "cos_time", "masked_CHL", "prev_day_CHL", "next_day-CHL"]:
        trd = ch[k][train_slice]; m, s = float(np.nanmean(trd)), float(np.nanstd(trd))
        stats[k] = (m, s); ch[k] = np.nan_to_num((ch[k] - m) / s, nan=0.0)
    if standardize_chl:
        trd = chl_log[train_slice]; ym, ys = float(np.nanmean(trd)), float(np.nanstd(trd))
    else:
        ym, ys = 0.0, 1.0
    y = (chl_log - ym) / ys
    stats["CHL"] = (ym, ys)
    return ch, y, stats


## 3. Split and train

Contiguous 60/20/20 split (a blocked alternative is in the sweep section), then a random-crop U-Net (40x56
tiles at batch 16 - the whole region is too big for batch training on one GPU) with a **masked loss** that
scores only real-ocean pixels.

In [6]:
CH_ORDER = ["sin_time", "cos_time", "masked_CHL", "prev_day_CHL", "next_day-CHL",
            "land_flag", "real_cloud_flag", "valid_CHL_flag", "fake_cloud_flag"]
T = chl.shape[0]
n_tr, n_va, buf = int(0.6 * T), int(0.2 * T), 2
tr = slice(0, n_tr); va = slice(n_tr + buf, n_tr + buf + n_va); te = slice(n_tr + buf + n_va + buf, T)
ch, y, stats = build_pace_channels(chl, times, tr, cloud_len=3, standardize_chl=True)
X = np.stack([ch[k] for k in CH_ORDER], -1).astype("float32")
y = y[..., None].astype("float32")                       # NaN kept on purpose
y_mean, y_std = stats["CHL"]
print("X:", X.shape, " train/val/test:", n_tr, n_va, T - te.start)


In [7]:
import os; os.makedirs("models/pace", exist_ok=True)

def masked_mse(y_true, y_pred):
    mask = tf.cast(tf.math.is_finite(y_true), tf.float32)
    yt = tf.where(tf.math.is_finite(y_true), y_true, 0.0)     # sanitize NaN so gradients stay clean
    return tf.reduce_sum(tf.square(yt - y_pred) * mask) / (tf.reduce_sum(mask) + 1e-6)

def masked_mae(y_true, y_pred):
    mask = tf.cast(tf.math.is_finite(y_true), tf.float32)
    yt = tf.where(tf.math.is_finite(y_true), y_true, 0.0)
    return tf.reduce_sum(tf.abs(yt - y_pred) * mask) / (tf.reduce_sum(mask) + 1e-6)

th, tw, npd, batch, NC = 40, 56, 16, 16, X.shape[-1]
def crop_gen(Xf, yf, seed):
    rng = np.random.default_rng(seed)
    def g():
        D, H, W, _ = Xf.shape
        for d in range(D):
            for _ in range(npd):
                yy = int(rng.integers(0, H - th + 1)); xx = int(rng.integers(0, W - tw + 1))
                yield Xf[d, yy:yy+th, xx:xx+tw], yf[d, yy:yy+th, xx:xx+tw]
    return g
sig = (tf.TensorSpec((th, tw, NC), tf.float32), tf.TensorSpec((th, tw, 1), tf.float32))
ds_tr = tf.data.Dataset.from_generator(crop_gen(X[tr], y[tr], 0), output_signature=sig
    ).shuffle(512, seed=0).batch(batch).repeat().prefetch(tf.data.AUTOTUNE)
ds_va = tf.data.Dataset.from_generator(crop_gen(X[va], y[va], 1), output_signature=sig
    ).batch(batch).repeat().prefetch(tf.data.AUTOTUNE)
steps_tr = (X[tr].shape[0] * npd) // batch; steps_va = (X[va].shape[0] * npd) // batch

model = mtg.UNet((None, None, NC))
model.compile(optimizer="adam", loss=masked_mse, metrics=[masked_mae])   # only real-ocean pixels count
es = tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
model.fit(ds_tr, epochs=50, steps_per_epoch=steps_tr,
          validation_data=ds_va, validation_steps=steps_va, callbacks=[es], verbose=2)
model.save("models/pace/pace_arabsea_v2.keras")


## 4. Evaluate (self-supervised, no Copernicus)

Fake-cloud MAE at the held-out pixels, plus two baselines: the training-period mean, and **persistence**
(previous day's value). A useful model beats both.

In [8]:
# Self-supervised evaluation: error only at the held-out fake-cloud pixels.
FCF = CH_ORDER.index("fake_cloud_flag")

def fake_cloud_mae(model, Xs, ys):
    pred = model.predict(Xs, batch_size=1, verbose=0)[..., 0] * y_std + y_mean
    truth = ys[..., 0] * y_std + y_mean
    m = (Xs[..., FCF] == 1) & np.isfinite(truth) & np.isfinite(pred)
    return float(np.mean(np.abs(pred[m] - truth[m])))

print("test fake-cloud MAE:", round(fake_cloud_mae(model, X[te], y[te]), 4))

# quick look at one test day: masked input, prediction, and observed truth
d = te.start + (T - te.start) // 2
pred = model.predict(X[d][None], verbose=0)[0, ..., 0] * y_std + y_mean
truth = y[d, ..., 0] * y_std + y_mean
masked_in = np.where(X[d, ..., CH_ORDER.index("masked_CHL")] == 0, np.nan,
                     X[d, ..., CH_ORDER.index("masked_CHL")])
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, arr, t in zip(ax, [masked_in, pred, truth], ["masked input", "U-Net prediction", "observed (truth)"]):
    im = a.imshow(arr, origin="upper", interpolation="nearest"); a.set_title(t); fig.colorbar(im, ax=a, shrink=0.7)
plt.suptitle(str(pd.to_datetime(times[d]).date())); plt.tight_layout(); plt.show()

In [9]:
FCF = CH_ORDER.index("fake_cloud_flag")
fake = X[te][..., FCF] == 1
truth = y[te][..., 0] * y_std + y_mean

# baseline 1: predict the training mean (the floor -- model must beat this easily)
mean_pred = float(np.nanmean(chl[tr]))
m = fake & np.isfinite(truth)
print("mean baseline MAE: ", round(np.mean(np.abs(mean_pred - truth[m])), 4))

# baseline 2: persistence -- yesterday's observed value (a STRONG baseline)
prev = np.roll(chl, 1, axis=0)[te]
mp = fake & np.isfinite(truth) & np.isfinite(prev)
print("persistence MAE:   ", round(np.mean(np.abs(prev[mp] - truth[mp])), 4))

# the U-Net
pred = model.predict(X[te], batch_size=1, verbose=0)[..., 0] * y_std + y_mean
mm = fake & np.isfinite(truth) & np.isfinite(pred)
print("U-Net MAE:         ", round(np.mean(np.abs(pred[mm] - truth[mm])), 4))
print("n fake-cloud pixels:", int(mm.sum()))


## 5. Diagnostics and visualization

Prediction panels, error over time, and per-pixel / per-month error maps. The coverage cell reports how
much of the ocean the fake clouds mask - if it is too high, lower `max_blob_frac`.

In [10]:
import cartopy.crs as ccrs, cartopy.feature as cfeature
FCF = CH_ORDER.index("fake_cloud_flag")
ext = [float(zarr_ds.lon.min()), float(zarr_ds.lon.max()),
       float(zarr_ds.lat.min()), float(zarr_ds.lat.max())]

preds = model.predict(X, batch_size=1, verbose=0)[..., 0] * y_std + y_mean   # (T,H,W) whole-region
truth = y[..., 0] * y_std + y_mean
fake  = X[..., FCF] == 1
valid = fake & np.isfinite(truth) & np.isfinite(preds)
ad = np.where(valid, np.abs(preds - truth), np.nan)                          # error only at fake-cloud pixels
mae_t = np.nanmean(ad.reshape(ad.shape[0], -1), axis=1)                       # per-step MAE
print("mean fake-cloud MAE:", round(float(np.nanmean(ad)), 4))


In [11]:
def pace_panel(t):
    tr_v = truth[t]; pr = np.where(np.isnan(tr_v), np.nan, preds[t])
    masked_in = np.where(fake[t], np.nan, tr_v)                # what the model saw (fake clouds hidden)
    vmin, vmax = np.nanmin(tr_v), np.nanmax(tr_v)
    fig, ax = plt.subplots(2, 2, figsize=(13, 9), subplot_kw={"projection": ccrs.PlateCarree()})
    def show(a, arr, ttl, **kw):
        im = a.imshow(arr, extent=ext, origin="upper", transform=ccrs.PlateCarree(),
                      interpolation="nearest", **kw)
        a.add_feature(cfeature.COASTLINE, linewidth=0.4); a.set_title(ttl, size=11)
        fig.colorbar(im, ax=a, shrink=0.6)
    show(ax[0,0], masked_in, "masked input (fake clouds hidden)", vmin=vmin, vmax=vmax)
    show(ax[0,1], fake[t].astype(float), "fake-cloud mask", vmin=0, vmax=1)
    show(ax[1,0], pr, "U-Net prediction", vmin=vmin, vmax=vmax)
    show(ax[1,1], tr_v - pr, "observed - predicted", vmin=-1, vmax=1, cmap=plt.cm.RdBu)
    plt.suptitle(f"PACE  {pd.to_datetime(times[t]).date()}"); plt.tight_layout(); plt.show()

cands = [i for i in range(te.start, X.shape[0]) if int(fake[i].sum()) > 500]   # test steps with real coverage
for t in (cands[:2] or [te.start]):
    pace_panel(t)


In [12]:
tt = pd.to_datetime(times)
is_tr = np.arange(len(tt)) < tr.stop
fig, a = plt.subplots(figsize=(12, 4))
a.plot(tt, mae_t, color="black", lw=1.2, zorder=1)
sel = is_tr & np.isfinite(mae_t)
a.plot(tt[sel], mae_t[sel], "o", ms=3, color="red", zorder=2, label="train")
a.axvline(tt[tr.stop], ls="--", c="gray", lw=0.8); a.axvline(tt[te.start], ls="--", c="gray", lw=0.8)
a.set_xlabel("time"); a.set_ylabel("fake-cloud MAE"); a.set_title("PACE gap-fill error over time")
a.legend(); a.grid(alpha=0.3); plt.show()


In [13]:
mae_map = np.nanmean(ad, axis=0)
vmax = float(np.nanpercentile(mae_map, 99))

fig, a = plt.subplots(figsize=(7, 5), subplot_kw={"projection": ccrs.PlateCarree()})
im = a.imshow(mae_map, vmin=0, vmax=vmax, extent=ext, origin="upper",
              transform=ccrs.PlateCarree(), interpolation="nearest")
a.add_feature(cfeature.COASTLINE, linewidth=0.4); fig.colorbar(im, ax=a, label="MAE")
a.set_title("PACE per-pixel gap-fill error (all time)"); plt.show()

months = pd.to_datetime(times).month
fig, axes = plt.subplots(3, 4, figsize=(14, 8), subplot_kw={"projection": ccrs.PlateCarree()})
for mo, a in zip(range(1, 13), axes.ravel()):
    sel = months == mo
    mm = np.nanmean(ad[sel], axis=0) if sel.any() else np.full(mae_map.shape, np.nan)
    im = a.imshow(mm, vmin=0, vmax=vmax, extent=ext, origin="upper",
                  transform=ccrs.PlateCarree(), interpolation="nearest")
    a.add_feature(cfeature.COASTLINE, linewidth=0.4); a.set_title(f"month {mo}", size=9)
fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.02, label="MAE"); plt.show()
plt.show()


In [14]:
FCF = CH_ORDER.index("fake_cloud_flag"); VC = CH_ORDER.index("valid_CHL_flag")
obs = np.isfinite(chl)
region = ~np.isnan(chl).all(0)                       # ocean (not always-land), (H,W)
fake_frac = (X[..., FCF] == 1).sum((1, 2)) / np.maximum(obs.sum((1, 2)), 1)
seen_frac = (X[..., VC] == 1).sum((1, 2)) / np.maximum(region.sum(), 1)
print(f"fake clouds cover {fake_frac.mean():.0%} of observed pixels (max {fake_frac.max():.0%})")
print(f"model sees {seen_frac.mean():.0%} of the ocean on average")

t = te.start + (X.shape[0] - te.start) // 2
mc = np.where(np.isnan(chl[t]) | (X[t, ..., FCF] == 1), np.nan, chl[t])   # what the model actually sees
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, arr, ttl in zip(ax, [chl[t], mc, X[t, ..., FCF].astype(float)],
                       ["observed (real gaps white)", "masked_CHL (model input)", "fake-cloud mask"]):
    im = a.imshow(arr, origin="upper"); a.set_title(ttl); fig.colorbar(im, ax=a, shrink=0.7)
plt.tight_layout(); plt.show()


## 6. Optional: experiment sweep (split x temporal window)

This section **redefines** `build_pace_channels` (fake clouds now filtered by **shape** - swaths are
long/thin/edge-spanning, dropped; large compact clouds kept) and adds a **season-aware blocked** split
(cut on the 4 Arabian Sea chlorophyll seasons, not calendar months). It then sweeps
{contiguous, season} x n_days {1,2,3} over several seeds, resumable via `pace_sweep_v2.pkl`. The
single-run pipeline above is enough to get started; this is for comparing configurations.

In [ ]:
from scipy import ndimage

# --- swath filter by SHAPE: keep compact blobs (clouds), drop long/thin/edge-to-edge bands (swaths) ---
def _keep_cloud_blobs(foot, H, W, span_frac=0.7, aspect=4.0):
    lab, n = ndimage.label(foot)
    if n == 0:
        return foot
    keep = np.zeros(n + 1, bool)
    for i, sl in enumerate(ndimage.find_objects(lab), start=1):
        if sl is None:
            continue
        h = sl[0].stop - sl[0].start; w = sl[1].stop - sl[1].start
        spans = (h >= span_frac * H) or (w >= span_frac * W)      # reaches edge-to-edge
        elongated = max(h, w) / max(1, min(h, w)) > aspect         # long and thin
        keep[i] = not (spans or elongated)                         # keep compact blobs, ANY size
    return keep[lab]

def build_pace_channels(chl_log, times, train_mask, n_days=1, cloud_len=3,
                        span_frac=0.7, aspect=4.0, standardize_chl=True):
    """n_days = # prev/next steps as channels (from masked CHL). Fake clouds mimic CLOUDS: swaths are
    dropped by SHAPE (long/thin/edge-spanning), so large compact monsoon clouds are kept. train_mask=bool[T]."""
    T, H, W = chl_log.shape
    gap = np.isnan(chl_log); land = gap.all(0); land3 = np.broadcast_to(land, chl_log.shape)
    real_cloud = gap & ~land3
    fake = np.zeros((T, H, W), bool)
    for s in range(0, T, cloud_len):
        e = min(s + cloud_len, T)
        foot = gap[(s + T // 2) % T] & ~land
        foot = _keep_cloud_blobs(foot, H, W, span_frac, aspect)    # <- shape filter (was size)
        fake[s:e] = foot[None] & ~gap[s:e] & ~land3[s:e]
    masked = np.where(fake, np.nan, chl_log)
    doy = pd.to_datetime(times).dayofyear.to_numpy().astype("float32"); ang = 2*np.pi*doy/365.0
    ch = {"sin_time": np.broadcast_to(np.sin(ang)[:,None,None], chl_log.shape).astype("float32").copy(),
          "cos_time": np.broadcast_to(np.cos(ang)[:,None,None], chl_log.shape).astype("float32").copy(),
          "masked_CHL": masked}
    for k in range(1, n_days + 1):
        p = np.roll(masked, k, 0);  p[:k] = np.nan
        nx = np.roll(masked, -k, 0); nx[-k:] = np.nan
        ch[f"prev{k}_CHL"] = p; ch[f"next{k}_CHL"] = nx
    numeric = list(ch.keys())
    valid = (~np.isnan(masked)) & ~land3
    ch.update({"land_flag": land3.astype("float32"), "real_cloud_flag": real_cloud.astype("float32"),
               "valid_CHL_flag": valid.astype("float32"), "fake_cloud_flag": fake.astype("float32")})
    stats = {}
    for key in numeric:
        trd = ch[key][train_mask]; m, s = float(np.nanmean(trd)), float(np.nanstd(trd))
        stats[key] = (m, s); ch[key] = np.nan_to_num((ch[key]-m)/s, nan=0.0)
    ym, ys = (float(np.nanmean(chl_log[train_mask])), float(np.nanstd(chl_log[train_mask]))) \
             if standardize_chl else (0.0, 1.0)
    stats["CHL"] = (ym, ys)
    order = numeric + ["land_flag", "real_cloud_flag", "valid_CHL_flag", "fake_cloud_flag"]
    return ch, (chl_log - ym) / ys, stats, order

# --- splits ---
def contiguous_split(T, buf=2):
    n_tr, n_va = int(0.6*T), int(0.2*T)
    tr = np.zeros(T, bool); va = np.zeros(T, bool); te = np.zeros(T, bool)
    tr[:n_tr] = True; va[n_tr+buf:n_tr+buf+n_va] = True; te[n_tr+buf+n_va+buf:] = True
    return tr, va, te

SEASON_BREAKS = [40, 75, 225, 305]   # day-of-year cuts between the 4 Arabian Sea chl seasons (eyeballed, tunable)
def season_of(doy):
    if SEASON_BREAKS[0] <= doy < SEASON_BREAKS[1]: return 0   # winter / NE-monsoon bloom
    if SEASON_BREAKS[1] <= doy < SEASON_BREAKS[2]: return 1   # spring inter-monsoon low
    if SEASON_BREAKS[2] <= doy < SEASON_BREAKS[3]: return 2   # summer / SW-monsoon bloom
    return 3                                                   # fall transition (wraps the year)

def season_blocked_split(times, buf=2):
    """Blocks = contiguous season runs. First occurrence of each season -> train (train sees all four);
    later occurrences rotate val/test (so val/test span several regimes and no block is split mid-season)."""
    t = pd.to_datetime(times); doy = t.dayofyear.to_numpy()
    seas = np.array([season_of(d) for d in doy])
    block = np.cumsum(np.r_[0, seas[1:] != seas[:-1]])       # contiguous-season block id
    role = np.array(["tr"] * len(seas), dtype=object)
    seen = {}; rot = 0
    for bid in range(block.max() + 1):
        m = block == bid; s = int(seas[m][0])
        c = seen.get(s, 0); seen[s] = c + 1
        if c > 0:
            role[m] = ("val", "test")[rot % 2]; rot += 1
    drop = np.array([len(set(role[max(0,i-buf):i+buf+1])) > 1 for i in range(len(role))])
    return (role=="tr") & ~drop, (role=="val") & ~drop, (role=="test") & ~drop

In [ ]:
# sanity-check the season split + the new (shape-filtered) coverage before the long sweep
trm, vam, tem = season_blocked_split(times)
print("season split sizes  train/val/test:", int(trm.sum()), int(vam.sum()), int(tem.sum()))
print("season of first steps:", [season_of(d) for d in pd.to_datetime(times).dayofyear[:8].tolist()], "...")

ch, _, _, order = build_pace_channels(chl, times, trm, n_days=1)
Xd = np.stack([ch[k] for k in order], -1).astype("float32")
FCF, VC = order.index("fake_cloud_flag"), order.index("valid_CHL_flag")
region = int((~np.isnan(chl).all(0)).sum())
print(f"fake clouds cover {(Xd[...,FCF]==1).sum()/max(1,np.isfinite(chl).sum()):.0%} of observed pixels")
print(f"model sees {(Xd[...,VC]==1).sum()/(region*chl.shape[0]):.0%} of the ocean on average")
t = int(tem.argmax())
plt.figure(figsize=(6,4)); plt.imshow(Xd[t,...,FCF], origin="upper")
plt.title(f"shape-filtered fake clouds  {pd.to_datetime(times[t]).date()}"); plt.colorbar(); plt.show()

In [ ]:
import gc, os, pickle
SEEDS = [0, 1, 2]
RES = "pace_sweep_v2.pkl"     # NEW file - build + split changed, so the old results are stale
results = pickle.load(open(RES, "rb")) if os.path.exists(RES) else {}

def masked_mse(yt, yp):
    m = tf.cast(tf.math.is_finite(yt), tf.float32); v = tf.where(tf.math.is_finite(yt), yt, 0.0)
    return tf.reduce_sum(tf.square(v - yp) * m) / (tf.reduce_sum(m) + 1e-6)
def masked_mae(yt, yp):
    m = tf.cast(tf.math.is_finite(yt), tf.float32); v = tf.where(tf.math.is_finite(yt), yt, 0.0)
    return tf.reduce_sum(tf.abs(v - yp) * m) / (tf.reduce_sum(m) + 1e-6)
def crop_gen(Xf, yf, npd, th, tw, seed):
    rng = np.random.default_rng(seed)
    def g():
        D, H, W, _ = Xf.shape
        for d in range(D):
            for _ in range(npd):
                yy = int(rng.integers(0, H-th+1)); xx = int(rng.integers(0, W-tw+1))
                yield Xf[d, yy:yy+th, xx:xx+tw], yf[d, yy:yy+th, xx:xx+tw]
    return g

def run_config(split, n_days, seed, th=40, tw=56, npd=16, batch=16):
    tf.keras.utils.set_random_seed(seed)
    trm, vam, tem = contiguous_split(chl.shape[0]) if split == "contig" else season_blocked_split(times)
    ch, y_, stats, order = build_pace_channels(chl, times, trm, n_days=n_days)
    Xc = np.stack([ch[k] for k in order], -1).astype("float32"); yc = y_[..., None].astype("float32")
    del ch; gc.collect()
    ym, ys = stats["CHL"]; FCF = order.index("fake_cloud_flag"); NC = Xc.shape[-1]
    sig = (tf.TensorSpec((th,tw,NC), tf.float32), tf.TensorSpec((th,tw,1), tf.float32))
    dtr = tf.data.Dataset.from_generator(crop_gen(Xc[trm], yc[trm], npd, th, tw, seed),
            output_signature=sig).shuffle(512, seed=seed).batch(batch).repeat().prefetch(tf.data.AUTOTUNE)
    dva = tf.data.Dataset.from_generator(crop_gen(Xc[vam], yc[vam], npd, th, tw, seed+100),
            output_signature=sig).batch(batch).repeat().prefetch(tf.data.AUTOTUNE)
    model = mtg.UNet((None, None, NC)); model.compile("adam", loss=masked_mse, metrics=[masked_mae])
    es = tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)
    model.fit(dtr, epochs=50, steps_per_epoch=max(1,(int(trm.sum())*npd)//batch),
              validation_data=dva, validation_steps=max(1,(int(vam.sum())*npd)//batch),
              callbacks=[es], verbose=0)
    preds = model.predict(Xc[tem], batch_size=1, verbose=0)[..., 0]*ys + ym
    truth = yc[tem][..., 0]*ys + ym; fake = Xc[tem][..., FCF] == 1
    prev_true = np.roll(chl, 1, 0)[tem]
    m = fake & np.isfinite(truth) & np.isfinite(preds); mp = fake & np.isfinite(truth) & np.isfinite(prev_true)
    u = float(np.mean(np.abs(preds[m]-truth[m]))); p = float(np.mean(np.abs(prev_true[mp]-truth[mp])))
    del model, Xc, yc; gc.collect()
    return u, p, int(m.sum())

for split in ["contig", "season"]:
    for nd in [1, 2, 3]:
        for seed in SEEDS:
            if (split, nd, seed) in results: continue
            try:
                u, p, n = run_config(split, nd, seed); results[(split, nd, seed)] = (u, p, n)
                pickle.dump(results, open(RES, "wb"))
                print(f"  {split:>7} n_days={nd} seed={seed}: U-Net {u:.4f}", flush=True)
            except Exception as ex:
                print(f"  {split:>7} n_days={nd} seed={seed}: FAILED {type(ex).__name__}: {ex}", flush=True)

print(f"\n{'split':>7} {'n_days':>6} {'U-Net mean':>11} {'std':>7} {'persist':>8} {'win':>4} {'seeds':>6}")
for split in ["contig", "season"]:
    for nd in [1, 2, 3]:
        got = [results[(split,nd,s)] for s in SEEDS if (split,nd,s) in results]
        if not got: continue
        vals = [g[0] for g in got]; pv = got[0][1]
        print(f"{split:>7} {nd:>6} {np.mean(vals):11.4f} {np.std(vals):7.4f} {pv:8.4f} "
              f"{'yes' if np.mean(vals)<pv else 'no':>4} {len(vals):>6}")

## 7. Unattended sweep, then figures

The in-notebook sweep above trains many models back-to-back in one kernel; on a T4 that eventually hits
GPU-memory / cuDNN failures and needs babysitting. The robust way is **one subprocess per config** - each
run gets a fresh GPU and releases it on exit, and each result is checkpointed to `pace_results/`, so the
whole sweep is resumable and unattended (re-run the driver and it skips what is done).

Two T4 landmines this works around (both throw `CUDNN_STATUS_EXECUTION_FAILED`, but only in a fresh
subprocess - the notebook kernel happens to tolerate them): a **custom Keras metric**, and **NaN in the
target tensor**. So `train_pace.py` uses a 2-channel `[value, mask]` target (no NaN reaches the graph) and
no custom metric. Run the writefile cell -> the driver (walk away) -> the table.

In [ ]:
%%writefile train_pace.py
import os, argparse, json
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import numpy as np, pandas as pd, xarray as xr, tensorflow as tf
from scipy import ndimage
from scipy.ndimage import gaussian_filter
import mindthegap as mtg

def _keep_cloud_blobs(foot, H, W, span_frac=0.7, aspect=4.0):
    lab, n = ndimage.label(foot)
    if n == 0: return foot
    keep = np.zeros(n + 1, bool)
    for i, sl in enumerate(ndimage.find_objects(lab), start=1):
        if sl is None: continue
        h = sl[0].stop - sl[0].start; w = sl[1].stop - sl[1].start
        keep[i] = not ((h >= span_frac*H) or (w >= span_frac*W) or max(h,w)/max(1,min(h,w)) > aspect)
    return keep[lab]

def synthetic_cloud_mask(H, W, coverage, blob_sigma, rng):
    f = gaussian_filter(rng.standard_normal((H, W)), blob_sigma)
    return f > np.quantile(f, 1.0 - coverage)

def build_pace_channels(chl_log, times, train_mask, n_days=1, cloud_len=3, cloud_mode="synthetic",
                        coverage=0.4, blob_sigma=6.0, span_frac=0.7, aspect=4.0, seed=0, standardize_chl=True):
    T, H, W = chl_log.shape
    gap = np.isnan(chl_log); land = gap.all(0); land3 = np.broadcast_to(land, chl_log.shape)
    real_cloud = gap & ~land3
    fake = np.zeros((T, H, W), bool); rng = np.random.default_rng(seed)
    for s in range(0, T, cloud_len):
        e = min(s + cloud_len, T)
        foot = (synthetic_cloud_mask(H, W, coverage, blob_sigma, rng) & ~land) if cloud_mode == "synthetic" \
               else _keep_cloud_blobs(gap[(s + T // 2) % T] & ~land, H, W, span_frac, aspect)
        fake[s:e] = foot[None] & ~gap[s:e] & ~land3[s:e]
    masked = np.where(fake, np.nan, chl_log)
    doy = pd.to_datetime(times).dayofyear.to_numpy().astype("float32"); ang = 2*np.pi*doy/365.0
    ch = {"sin_time": np.broadcast_to(np.sin(ang)[:,None,None], chl_log.shape).astype("float32").copy(),
          "cos_time": np.broadcast_to(np.cos(ang)[:,None,None], chl_log.shape).astype("float32").copy(),
          "masked_CHL": masked}
    for k in range(1, n_days + 1):
        p = np.roll(masked, k, 0);  p[:k] = np.nan
        nx = np.roll(masked, -k, 0); nx[-k:] = np.nan
        ch[f"prev{k}_CHL"] = p; ch[f"next{k}_CHL"] = nx
    numeric = list(ch.keys()); valid = (~np.isnan(masked)) & ~land3
    ch.update({"land_flag": land3.astype("float32"), "real_cloud_flag": real_cloud.astype("float32"),
               "valid_CHL_flag": valid.astype("float32"), "fake_cloud_flag": fake.astype("float32")})
    stats = {}
    for key in numeric:
        trd = ch[key][train_mask]; m, s = float(np.nanmean(trd)), float(np.nanstd(trd))
        stats[key] = (m, s); ch[key] = np.nan_to_num((ch[key]-m)/(s+1e-8), nan=0.0, posinf=0.0, neginf=0.0)
    ym, ys = (float(np.nanmean(chl_log[train_mask])), float(np.nanstd(chl_log[train_mask]))) if standardize_chl else (0.0, 1.0)
    stats["CHL"] = (ym, ys)
    order = numeric + ["land_flag", "real_cloud_flag", "valid_CHL_flag", "fake_cloud_flag"]
    return ch, (chl_log - ym) / ys, stats, order

SEASON_BREAKS = [40, 75, 225, 305]
def season_of(d):
    if SEASON_BREAKS[0] <= d < SEASON_BREAKS[1]: return 0
    if SEASON_BREAKS[1] <= d < SEASON_BREAKS[2]: return 1
    if SEASON_BREAKS[2] <= d < SEASON_BREAKS[3]: return 2
    return 3
def season_blocked_split(times, buf=2):
    doy = pd.to_datetime(times).dayofyear.to_numpy()
    seas = np.array([season_of(d) for d in doy])
    block = np.cumsum(np.r_[0, seas[1:] != seas[:-1]])
    role = np.array(["tr"]*len(seas), dtype=object); seen = {}; rot = 0
    for bid in range(block.max()+1):
        mm = block == bid; s = int(seas[mm][0]); c = seen.get(s, 0); seen[s] = c + 1
        if c > 0: role[mm] = ("val", "test")[rot % 2]; rot += 1
    drop = np.array([len(set(role[max(0,i-buf):i+buf+1])) > 1 for i in range(len(role))])
    return (role=="tr")&~drop, (role=="val")&~drop, (role=="test")&~drop

def masked_mse(yt, yp):
    y_true = yt[..., 0:1]; mask = yt[..., 1:2]
    return tf.reduce_sum(tf.square(y_true - yp) * mask) / (tf.reduce_sum(mask) + 1e-6)

def make_crops(Xf, Yf, npd, th, tw, seed):
    rng = np.random.default_rng(seed); D, H, W = Xf.shape[:3]
    xs, ys_ = [], []
    for d in range(D):
        for _ in range(npd):
            yy = int(rng.integers(0, H-th+1)); xx = int(rng.integers(0, W-tw+1))
            xs.append(Xf[d, yy:yy+th, xx:xx+tw]); ys_.append(Yf[d, yy:yy+th, xx:xx+tw])
    return np.stack(xs).astype("float32"), np.stack(ys_).astype("float32")

ap = argparse.ArgumentParser()
ap.add_argument("--method", required=True); ap.add_argument("--n_days", type=int, required=True)
ap.add_argument("--seed", type=int, required=True); ap.add_argument("--cache", default="pace_arabsea_3day.zarr")
ap.add_argument("--out", required=True)
a = ap.parse_args()
for g in tf.config.list_physical_devices("GPU"): tf.config.experimental.set_memory_growth(g, True)
tf.keras.utils.set_random_seed(a.seed)

zarr_ds = xr.open_zarr(a.cache)
chl = np.log(zarr_ds["chlor_a"].where(zarr_ds["chlor_a"] > 0).load().values).astype("float32")
times = zarr_ds.time.values

trm, vam, tem = season_blocked_split(times)
kw = dict(cloud_mode="shape") if a.method == "shape" else dict(cloud_mode="synthetic", coverage=int(a.method[5:])/100)
ch, y_, stats, order = build_pace_channels(chl, times, trm, n_days=a.n_days, seed=a.seed, **kw)
X = np.stack([ch[k] for k in order], -1).astype("float32")
ym, ys = stats["CHL"]
Y2 = np.stack([np.nan_to_num(y_, nan=0.0), np.isfinite(y_).astype("float32")], -1).astype("float32")
NC = X.shape[-1]; th, tw, npd, batch = 40, 56, 16, 16

Xtr, Ytr = make_crops(X[trm], Y2[trm], npd, th, tw, a.seed)
Xva, Yva = make_crops(X[vam], Y2[vam], npd, th, tw, a.seed + 100)
dtr = tf.data.Dataset.from_tensor_slices((Xtr, Ytr)).shuffle(min(1024, len(Xtr)), seed=a.seed).batch(batch).repeat().prefetch(tf.data.AUTOTUNE)
dva = tf.data.Dataset.from_tensor_slices((Xva, Yva)).batch(batch).repeat().prefetch(tf.data.AUTOTUNE)

model = mtg.UNet((None, None, NC)); model.compile("adam", loss=masked_mse)   # no custom metric (T4 cuDNN fix)
es = tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)
model.fit(dtr, epochs=50, steps_per_epoch=max(1, len(Xtr)//batch),
          validation_data=dva, validation_steps=max(1, len(Xva)//batch), callbacks=[es], verbose=2)

preds = model.predict(X[tem], batch_size=1, verbose=0)[..., 0] * ys + ym
truth = chl[tem]; fake = X[tem][..., order.index("fake_cloud_flag")] == 1; prev = np.roll(chl, 1, 0)[tem]
m = fake & np.isfinite(truth) & np.isfinite(preds); mp = fake & np.isfinite(truth) & np.isfinite(prev)
u = float(np.mean(np.abs(preds[m]-truth[m]))); p = float(np.mean(np.abs(prev[mp]-truth[mp])))
json.dump({"method": a.method, "n_days": a.n_days, "seed": a.seed, "unet": u, "persist": p,
           "n": int(m.sum()), "cov": float(fake.sum()/max(1, np.isfinite(truth).sum()))}, open(a.out, "w"))

if os.environ.get("SAVE_VIZ"):     # rich arrays for the figures (log-chl units)
    np.savez_compressed(os.environ["SAVE_VIZ"],
        preds=preds.astype("float32"), truth=truth.astype("float32"), prev=prev.astype("float32"),
        fake=fake, land=np.isnan(chl).all(0),
        times=np.array([str(t)[:10] for t in pd.to_datetime(times[tem])]),
        method=a.method, n_days=a.n_days, cov=float(fake.sum()/max(1, np.isfinite(truth).sum())))
print("DONE", a.method, a.n_days, a.seed, "-> U-Net", round(u, 4), "persist", round(p, 4))


In [ ]:
import os, sys, subprocess, json
os.makedirs("pace_results", exist_ok=True)
for method in ["synth20", "synth40", "shape"]:
    for nd in [1, 2, 3, 4]:
        for seed in [0, 1, 2]:
            out = f"pace_results/{method}_nd{nd}_seed{seed}.json"
            if os.path.exists(out):
                continue
            print(f"===== {method} n_days={nd} seed={seed} =====", flush=True)
            rc = subprocess.run([sys.executable, "train_pace.py", "--method", method,
                                 "--n_days", str(nd), "--seed", str(seed), "--out", out]).returncode
            if os.path.exists(out):
                r = json.load(open(out)); print(f"  U-Net {r['unet']:.4f}  persist {r['persist']:.4f}  cov {r['cov']:.0%}")
            else:
                print(f"  exit {rc} - will retry on next run")
print("\nALL DONE")

In [ ]:
import glob, json, numpy as np
rows = [json.load(open(f)) for f in glob.glob("pace_results/*.json")]
print(f"configs finished: {len(rows)}\n")
print(f"{'method':>8}{'n_days':>7}{'U-Net':>9}{'std':>7}{'persist':>9}{'win':>5}{'cov':>6}")
for meth in sorted({r['method'] for r in rows}):
    for nd in sorted({r['n_days'] for r in rows if r['method'] == meth}):
        g = [r for r in rows if r['method'] == meth and r['n_days'] == nd]
        us = [r['unet'] for r in g]; ps = [r['persist'] for r in g]
        win = 'yes' if np.mean(us) < np.mean(ps) else 'no'
        print(f"{meth:>8}{nd:>7}{np.mean(us):9.3f}{np.std(us):7.3f}{np.mean(ps):9.3f}{win:>5}{np.mean([r['cov'] for r in g]):6.0%}")

### Figures for the writeup

`SAVE_VIZ` trains one representative model (40% synthetic clouds, 1-day window) and dumps the test-set
prediction arrays to `pace_viz.npz`. Run it on its own - not while the sweep is going, or the two
trainings fight over the one GPU. Everything after reads only `pace_viz.npz` (and the cached zarr for the
cloud figure), no GPU.

In [ ]:
import os
os.environ["SAVE_VIZ"] = "pace_viz.npz"     # set here; the `!VAR=... ` prefix does not propagate through Jupyter's shell
!python train_pace.py --method synth40 --n_days 1 --seed 0 --out pace_results/synth40_nd1_seed0.json

In [ ]:
import numpy as np, matplotlib.pyplot as plt
d = np.load("pace_viz.npz", allow_pickle=True)
preds, truth, fake, prev = d["preds"], d["truth"], d["fake"], d["prev"]
land, times = d["land"], d["times"]
m  = fake & np.isfinite(truth) & np.isfinite(preds)
mp = fake & np.isfinite(truth) & np.isfinite(prev)
mae_u, mae_p = np.mean(np.abs(preds[m]-truth[m])), np.mean(np.abs(prev[mp]-truth[mp]))
rel = np.abs(np.exp(preds[m]) - np.exp(truth[m])) / np.exp(truth[m])          # relative error, real chl units
print(f"Held-out cloud pixels scored : {m.sum():,}   (fake-cloud coverage {float(d['cov']):.0%})")
print(f"U-Net    log-MAE : {mae_u:.3f}   (~{np.exp(mae_u)-1:.0%} typical multiplicative error)")
print(f"Persist  log-MAE : {mae_p:.3f}   (~{np.exp(mae_p)-1:.0%})")
print(f"U-Net beats persistence by      : {100*(mae_p-mae_u)/mae_p:.0f}%")
print(f"Median % error (real chl units) : {np.median(rel):.0%}    Mean % error : {np.mean(rel):.0%}")

In [ ]:
idx = np.random.default_rng(0).choice(m.sum(), size=min(20000, m.sum()), replace=False)
pl, tl = preds[m][idx], truth[m][idx]
fig, ax = plt.subplots(1, 2, figsize=(12, 5.5))
ax[0].scatter(tl, pl, s=2, alpha=0.12)
lo, hi = np.percentile(np.r_[tl, pl], [0.5, 99.5]); ax[0].plot([lo, hi], [lo, hi], "r-", lw=1)
ax[0].set(xlabel="observed log-chl", ylabel="predicted log-chl", title="Log space")
te, pe = np.exp(tl), np.exp(pl)
ax[1].scatter(te, pe, s=2, alpha=0.12); ax[1].plot([te.min(), te.max()], [te.min(), te.max()], "r-", lw=1)
ax[1].set(xlabel="observed chl (mg/m3)", ylabel="predicted chl", title="Linear space", xscale="log", yscale="log")
plt.suptitle("U-Net vs observed at held-out cloud pixels (1:1 = red)"); plt.tight_layout(); plt.show()

In [ ]:
log_err = preds[m] - truth[m]
rel_err = (np.exp(preds[m]) - np.exp(truth[m])) / np.exp(truth[m])
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].hist(log_err, bins=80, range=(-1, 1)); ax[0].axvline(0, color="r")
ax[0].set(title=f"Log error (pred - obs)   MAE={np.mean(np.abs(log_err)):.3f}", xlabel="log-chl error")
ax[1].hist(np.clip(rel_err, -1, 2), bins=80); ax[1].axvline(0, color="r")
ax[1].set(title=f"Relative error   median|.|={np.median(np.abs(rel_err)):.0%}", xlabel="(pred - obs)/obs")
plt.tight_layout(); plt.show()

In [ ]:
gappy = np.argsort(-((np.isnan(truth)) & ~land).sum((1, 2)))[:3]     # days with the biggest real gaps
cm = plt.cm.viridis.copy(); cm.set_bad("lightgray")
for dd in gappy:
    saw = np.where(fake[dd], np.nan, truth[dd])       # what the model saw (real gaps + fake clouds = holes)
    vmin, vmax = np.nanpercentile(truth[dd], [2, 98])
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
    for a_, arr, t in zip(ax, [saw, preds[dd], truth[dd]],
                          ["observed (gaps hidden)", "U-Net gap-filled", "actual observed"]):
        im = a_.imshow(arr, origin="upper", cmap=cm, vmin=vmin, vmax=vmax); a_.set_title(t); a_.axis("off")
        fig.colorbar(im, ax=a_, shrink=0.7, label="log-chl")
    plt.suptitle(f"Arabian Sea chlorophyll - {times[dd]}"); plt.tight_layout(); plt.show()

In [ ]:
import numpy as np, xarray as xr, matplotlib.pyplot as plt
from scipy import ndimage
from scipy.ndimage import gaussian_filter
z = xr.open_zarr("pace_arabsea_3day.zarr")
chl = np.log(z["chlor_a"].where(z["chlor_a"] > 0).load().values).astype("float32")
gap = np.isnan(chl); land = gap.all(0); H, W = land.shape; oc = ~land
di = int(np.argsort((gap & ~land).sum((1, 2)))[len(chl)//2])     # a median-gappy day
real = gap[di] & ~land
def keep(foot, span=0.7, asp=4.0):
    lab, n = ndimage.label(foot); k = np.zeros(n+1, bool)
    for i, sl in enumerate(ndimage.find_objects(lab), 1):
        if sl is None: continue
        h = sl[0].stop-sl[0].start; w = sl[1].stop-sl[1].start
        k[i] = not ((h >= span*H) or (w >= span*W) or max(h, w)/max(1, min(h, w)) > asp)
    return k[lab]
shape_kept = keep(real)
rng = np.random.default_rng(0); f = gaussian_filter(rng.standard_normal((H, W)), 6.0)
synth = (f > np.quantile(f, 0.6)) & ~land
cov = lambda mm: (mm & oc).sum() / oc.sum()
fig, ax = plt.subplots(1, 4, figsize=(18, 4.2))
panels = [(chl[di], "observed chl", "viridis"),
          (real, f"real PACE gaps  {cov(real):.0%}", "gray_r"),
          (shape_kept, f"shape-filtered clouds  {cov(shape_kept):.0%}", "gray_r"),
          (synth, f"synthetic clouds  {cov(synth):.0%}", "gray_r")]
for a_, (arr, t, cmap) in zip(ax, panels):
    a_.imshow(arr, cmap=cmap); a_.set_title(t); a_.axis("off")
plt.suptitle("Real gaps are big orbital swaths; we train on smaller cloud-shaped holes"); plt.tight_layout(); plt.show()
print(f"Real PACE gaps cover ~{cov(real):.0%} of the ocean; our fake clouds cover ~{cov(synth):.0%}.")

### Four-panel maps per cloud method (observed / flags / prediction / difference)

The same 4-panel view as notebook 2, for the PACE model: observed log-chl, the land / fake-cloud /
observed flag map, the U-Net prediction, and observed-minus-predicted. Made for both cloud methods so we
can compare what each does. `nb7res-05` already made `pace_viz.npz` (synthetic 40%); the cell below makes
the shape-method version, then both are plotted on the same test day.

In [ ]:
import os
os.environ["SAVE_VIZ"] = "pace_viz_shape.npz"
!python train_pace.py --method shape --n_days 1 --seed 0 --out pace_results/shape_nd1_seed0.json

In [ ]:
import numpy as np, xarray as xr, matplotlib.pyplot as plt
import cartopy.crs as ccrs, cartopy.feature as cfeature

_z = xr.open_zarr("pace_arabsea_3day.zarr")
EXT = [float(_z.lon.min()), float(_z.lon.max()), float(_z.lat.min()), float(_z.lat.max())]
proj = ccrs.PlateCarree()

def pace_4panel(npz_path, title, day=None):
    d = np.load(npz_path, allow_pickle=True)
    preds, truth, fake, land, times = d["preds"], d["truth"], d["fake"], d["land"], d["times"]
    if day is None:
        day = int(np.argmax((fake & np.isfinite(truth)).sum((1, 2))))
    obs, pr, fk = truth[day], preds[day], fake[day]
    flags = np.where(np.isfinite(obs) & ~fk, 2.0, np.where(fk, 1.0, 0.0))   # 0 land/real-cloud, 1 fake, 2 observed
    vmin, vmax = np.nanpercentile(obs, [2, 98])
    panels = [(np.where(land, np.nan, obs),                "Observed Level-3 log Chl-a",      "viridis", vmin, vmax, "log Chl-a"),
              (flags,                                       "land/real-cloud=0, fake=1, obs=2", "viridis", 0,   2,    "flag"),
              (np.where(land, np.nan, pr),                  "Predicted log Chl-a from U-Net",  "viridis", vmin, vmax, "log Chl-a"),
              (np.where(np.isfinite(obs), obs - pr, np.nan), "Observed - Predicted (log)",      "RdBu",   -1,   1,    "diff log Chl-a")]
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), subplot_kw={"projection": proj})
    for a, (arr, ttl, cmap, vmn, vmx, clab) in zip(axes.ravel(), panels):
        im = a.imshow(arr, extent=EXT, origin="upper", transform=proj, cmap=cmap,
                      vmin=vmn, vmax=vmx, interpolation="nearest")
        a.add_feature(cfeature.COASTLINE, linewidth=0.5)
        gl = a.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4); gl.top_labels = False; gl.right_labels = False
        a.set_title(ttl, size=10); fig.colorbar(im, ax=a, shrink=0.7, label=clab)
    plt.suptitle(f"{title}   {times[day]}", size=13); plt.tight_layout(); plt.show()

ds = np.load("pace_viz.npz", allow_pickle=True)      # synthetic 40% (from nb7res-05)
day = int(np.argmax((ds["fake"] & np.isfinite(ds["truth"])).sum((1, 2))))   # same day for both -> comparable
pace_4panel("pace_viz.npz",       "Synthetic clouds (40%)", day=day)
pace_4panel("pace_viz_shape.npz", "Shape-filtered clouds",  day=day)

## Cloud temporal autocorrelation experiment

Real clouds persist day to day (autocorrelated), so a cloudy pixel is usually still cloudy in the
neighbouring composite -> the prev/next-day channels rarely reveal it and the model must fill it
*spatially*. Our synthetic clouds are (near-)independent per composite, so a hidden pixel is usually
visible in prev/next -> the model can just copy it forward. That inflates the fake-cloud score in a way
that will not transfer to real clouds.

Two fixes tested here: (1) drop prev/next (`n_days=0`), and (2) give the synthetic clouds real persistence
with a 3-D gaussian-noise cube (`time_sigma` = day-to-day persistence; 0 = independent, larger = clouds
that last and evolve). We first measure the real gap autocorrelation to calibrate `time_sigma`, then
compare {independent, correlated} x {prev/next on/off}. If prev/next helps a lot under *independent* clouds
but little under *correlated* ones, that gap is the size of the cheat. (The real-gap AC also includes
orbital swaths, which persist strongly, so treat it as an upper bound until we have PACE's swath flag.)

In [ ]:
import numpy as np, xarray as xr
from scipy.ndimage import gaussian_filter
z = xr.open_zarr("pace_arabsea_3day.zarr")
chl = np.log(z["chlor_a"].where(z["chlor_a"] > 0).load().values).astype("float32")
gap = np.isnan(chl); land = gap.all(0); ocean = ~land
g = gap[:, ocean].astype(float)                                   # (T, n_ocean) real gap mask
real_ac = np.corrcoef(g[:-1].ravel(), g[1:].ravel())[0, 1]
p_cc = (g[1:][g[:-1] == 1] == 1).mean(); base = g.mean()
print(f"real gap coverage {base:.0%}, lag-1 gap autocorrelation {real_ac:.2f}")
print(f"P(cloudy next | cloudy now) {p_cc:.0%}  vs base rate {base:.0%}   (high = persistent gaps)\\n")

T, H, W = chl.shape
def cube_ac(ts, coverage=0.4, seed=0):                            # synthetic-cube lag-1 AC for a time_sigma
    f = gaussian_filter(np.random.default_rng(seed).standard_normal((T, H, W)), sigma=(ts, 6.0, 6.0))
    c = (f > np.quantile(f, 1 - coverage))[:, ocean].astype(float)
    return np.corrcoef(c[:-1].ravel(), c[1:].ravel())[0, 1]
print("time_sigma -> synthetic lag-1 AC (pick the one closest to the real AC above):")
best = None
for ts in [0, 0.5, 1.0, 1.5, 2.0, 3.0]:
    ac = cube_ac(ts); print(f"  time_sigma={ts}: {ac:.2f}")
    if best is None or abs(ac - real_ac) < abs(best[1] - real_ac): best = (ts, ac)
print(f"\\n-> use TS_CORR = {best[0]} in the driver (synthetic AC {best[1]:.2f} vs real {real_ac:.2f})")

In [ ]:
%%writefile train_pace_ac.py
"""One config for the cloud-autocorrelation experiment: 3-D correlated synthetic clouds (time_sigma) and
optional prev/next (n_days). Masked loss, fake-cloud eval -> JSON. cuDNN-safe (2-ch target, no metric)."""
import os, argparse, json
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import numpy as np, pandas as pd, xarray as xr, tensorflow as tf
from scipy.ndimage import gaussian_filter
import mindthegap as mtg

def masked_mse(yt, yp):
    y = yt[..., 0:1]; m = yt[..., 1:2]
    return tf.reduce_sum(tf.square(y - yp) * m) / (tf.reduce_sum(m) + 1e-6)

def build_channels(chl_log, times, train_mask, n_days, coverage, blob_sigma, time_sigma, seed):
    T, H, W = chl_log.shape
    gap = np.isnan(chl_log); land = gap.all(0); land3 = np.broadcast_to(land, chl_log.shape)
    real_cloud = gap & ~land3
    rng = np.random.default_rng(seed)
    f = gaussian_filter(rng.standard_normal((T, H, W)), sigma=(time_sigma, blob_sigma, blob_sigma))
    cube = f > np.quantile(f, 1.0 - coverage)                # temporally-correlated synthetic clouds
    fake = cube & ~gap & ~land3                              # hide only observed ocean pixels
    masked = np.where(fake, np.nan, chl_log)
    doy = pd.to_datetime(times).dayofyear.to_numpy().astype("float32"); ang = 2*np.pi*doy/365.0
    ch = {"sin_time": np.broadcast_to(np.sin(ang)[:,None,None], chl_log.shape).astype("float32").copy(),
          "cos_time": np.broadcast_to(np.cos(ang)[:,None,None], chl_log.shape).astype("float32").copy(),
          "masked_CHL": masked}
    for k in range(1, n_days + 1):
        p = np.roll(masked, k, 0);  p[:k] = np.nan
        nx = np.roll(masked, -k, 0); nx[-k:] = np.nan
        ch[f"prev{k}_CHL"] = p; ch[f"next{k}_CHL"] = nx
    numeric = list(ch.keys()); valid = (~np.isnan(masked)) & ~land3
    ch.update({"land_flag": land3.astype("float32"), "real_cloud_flag": real_cloud.astype("float32"),
               "valid_CHL_flag": valid.astype("float32"), "fake_cloud_flag": fake.astype("float32")})
    for key in numeric:
        trd = ch[key][train_mask]; m, s = float(np.nanmean(trd)), float(np.nanstd(trd))
        ch[key] = np.nan_to_num((ch[key]-m)/(s+1e-8), nan=0.0, posinf=0.0, neginf=0.0)
    ym, ys = float(np.nanmean(chl_log[train_mask])), float(np.nanstd(chl_log[train_mask]))
    order = numeric + ["land_flag", "real_cloud_flag", "valid_CHL_flag", "fake_cloud_flag"]
    return ch, (chl_log - ym) / ys, (ym, ys), order

SEASON_BREAKS = [40, 75, 225, 305]
def season_of(d):
    if SEASON_BREAKS[0] <= d < SEASON_BREAKS[1]: return 0
    if SEASON_BREAKS[1] <= d < SEASON_BREAKS[2]: return 1
    if SEASON_BREAKS[2] <= d < SEASON_BREAKS[3]: return 2
    return 3
def season_blocked_split(times, buf=2):
    doy = pd.to_datetime(times).dayofyear.to_numpy()
    seas = np.array([season_of(d) for d in doy])
    block = np.cumsum(np.r_[0, seas[1:] != seas[:-1]])
    role = np.array(["tr"]*len(seas), dtype=object); seen = {}; rot = 0
    for bid in range(block.max()+1):
        mm = block == bid; s = int(seas[mm][0]); c = seen.get(s, 0); seen[s] = c + 1
        if c > 0: role[mm] = ("val", "test")[rot % 2]; rot += 1
    drop = np.array([len(set(role[max(0,i-buf):i+buf+1])) > 1 for i in range(len(role))])
    return (role=="tr")&~drop, (role=="val")&~drop, (role=="test")&~drop

def make_crops(Xf, Yf, npd, th, tw, seed):
    rng = np.random.default_rng(seed); D, H, W = Xf.shape[:3]
    xs, ys_ = [], []
    for d in range(D):
        for _ in range(npd):
            yy = int(rng.integers(0, H-th+1)); xx = int(rng.integers(0, W-tw+1))
            xs.append(Xf[d, yy:yy+th, xx:xx+tw]); ys_.append(Yf[d, yy:yy+th, xx:xx+tw])
    return np.stack(xs).astype("float32"), np.stack(ys_).astype("float32")

ap = argparse.ArgumentParser()
ap.add_argument("--n_days", type=int, required=True)
ap.add_argument("--time_sigma", type=float, required=True)
ap.add_argument("--coverage", type=float, default=0.4)
ap.add_argument("--seed", type=int, default=0)
ap.add_argument("--cache", default="pace_arabsea_3day.zarr")
ap.add_argument("--out", required=True)
a = ap.parse_args()
for g in tf.config.list_physical_devices("GPU"): tf.config.experimental.set_memory_growth(g, True)
tf.keras.utils.set_random_seed(a.seed)

z = xr.open_zarr(a.cache)
chl = np.log(z["chlor_a"].where(z["chlor_a"] > 0).load().values).astype("float32")
times = z.time.values
trm, vam, tem = season_blocked_split(times)
ch, y_, (ym, ys), order = build_channels(chl, times, trm, a.n_days, a.coverage, 6.0, a.time_sigma, a.seed)
X = np.stack([ch[k] for k in order], -1).astype("float32")
Y2 = np.stack([np.nan_to_num(y_, nan=0.0), np.isfinite(y_).astype("float32")], -1).astype("float32")
NC = X.shape[-1]; th, tw, npd, batch = 40, 56, 16, 16

Xtr, Ytr = make_crops(X[trm], Y2[trm], npd, th, tw, a.seed)
Xva, Yva = make_crops(X[vam], Y2[vam], npd, th, tw, a.seed + 100)
dtr = tf.data.Dataset.from_tensor_slices((Xtr, Ytr)).shuffle(min(1024, len(Xtr)), seed=a.seed).batch(batch).repeat().prefetch(tf.data.AUTOTUNE)
dva = tf.data.Dataset.from_tensor_slices((Xva, Yva)).batch(batch).repeat().prefetch(tf.data.AUTOTUNE)
model = mtg.UNet((None, None, NC)); model.compile("adam", loss=masked_mse)
es = tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)
model.fit(dtr, epochs=50, steps_per_epoch=max(1, len(Xtr)//batch),
          validation_data=dva, validation_steps=max(1, len(Xva)//batch), callbacks=[es], verbose=0)
Xt = X[tem]                                              # per-frame predict (avoids one big GPU copy)
preds = np.stack([model.predict(Xt[i:i+1], verbose=0)[0, ..., 0] for i in range(Xt.shape[0])]) * ys + ym
truth = chl[tem]; fake = Xt[..., order.index("fake_cloud_flag")] == 1; prev = np.roll(chl, 1, 0)[tem]
m = fake & np.isfinite(truth) & np.isfinite(preds); mp = fake & np.isfinite(truth) & np.isfinite(prev)
u = float(np.mean(np.abs(preds[m]-truth[m]))); p = float(np.mean(np.abs(prev[mp]-truth[mp])))
json.dump({"n_days": a.n_days, "time_sigma": a.time_sigma, "coverage": a.coverage, "unet": u, "persist": p,
           "n": int(m.sum()), "cov": float(fake.sum()/max(1, np.isfinite(truth).sum()))}, open(a.out, "w"))
print("DONE n_days", a.n_days, "time_sigma", a.time_sigma, "-> U-Net", round(u, 4), "persist", round(p, 4))

In [ ]:
import os, sys, subprocess, json
os.makedirs("pace_ac", exist_ok=True)
TS_CORR = 1.5    # <-- set from the measurement cell's recommendation
COV = 0.4
configs = [(1, 0.0), (0, 0.0), (1, TS_CORR), (0, TS_CORR)]   # (n_days, time_sigma)
for nd, ts in configs:
    out = f"pace_ac/nd{nd}_ts{ts}.json"
    if os.path.exists(out):
        continue
    print(f"===== n_days={nd} time_sigma={ts} =====", flush=True)
    subprocess.run([sys.executable, "train_pace_ac.py", "--n_days", str(nd), "--time_sigma", str(ts),
                    "--coverage", str(COV), "--seed", "0", "--out", out])
    if os.path.exists(out):
        r = json.load(open(out)); print(f"  U-Net {r['unet']:.4f}  persist {r['persist']:.4f}")
print("done")

In [ ]:
import glob, json
rows = [json.load(open(f)) for f in glob.glob("pace_ac/*.json")]
tss = sorted(set(r["time_sigma"] for r in rows))
print(f"{'cloud regime':>18}  {'prev/next':>9}  {'U-Net':>7}  {'persist':>7}  {'cov':>5}")
for ts in tss:
    for nd in sorted(set(r["n_days"] for r in rows if r["time_sigma"] == ts)):
        r = [x for x in rows if x["time_sigma"] == ts and x["n_days"] == nd][0]
        reg = "independent" if ts == 0 else f"correlated ts={ts}"
        print(f"{reg:>18}  {('on' if nd else 'off'):>9}  {r['unet']:7.4f}  {r['persist']:7.4f}  {r['cov']:5.0%}")
print("\n--- how much prev/next helps under each cloud regime (big under independent = the cheat) ---")
for ts in tss:
    d = {r["n_days"]: r["unet"] for r in rows if r["time_sigma"] == ts}
    if 0 in d and 1 in d:
        reg = "independent" if ts == 0 else f"correlated ts={ts}"
        print(f"{reg:>18}: prev/next {d[0]-d[1]:+.4f}  ({100*(d[0]-d[1])/d[0]:+.0f}% of the no-prevnext error)")

## Slide figures (for the deck)

Self-contained figure cells (each has its own imports). Each saves a PNG. Data needs:
- **autoencoder, strategy bars, temporal leak**: nothing, pure matplotlib (numbers hardcoded from the runs).
- **coverage, error maps**: the CMEMS `IO.zarr`.
- **PACE clouds, 4-panel**: the cached `pace_arabsea_3day.zarr` / `pace_viz.npz`.
The **error-maps** cell imports TensorFlow and holds the GPU; run it on its own, and restart the kernel
before any subprocess sweep afterward.

In [12]:
# --- fig 1: autoencoder diagram ---
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch
xs = [0, 1.1, 2.2, 3.3, 4.4, 5.5, 6.6]
h  = [3.0, 2.2, 1.4, 0.8, 1.4, 2.2, 3.0]
col = ["#4C72B0"]*3 + ["#C44E52"] + ["#55A868"]*3
fig, ax = plt.subplots(figsize=(11, 4.6)); ax.axis("off")
for x, hh, c in zip(xs, h, col):
    ax.add_patch(Rectangle((x, -hh/2), 0.7, hh, facecolor=c, edgecolor="k", lw=1.2))
for i in range(len(xs)-1):
    ax.add_patch(FancyArrowPatch((xs[i]+0.7, 0), (xs[i+1], 0), arrowstyle="-|>", mutation_scale=14, color="0.45"))
ax.text(1.45, 2.1, "ENCODER\nshrink", ha="center", weight="bold", color="#4C72B0", fontsize=12)
ax.text(5.15, 2.1, "DECODER\nrebuild", ha="center", weight="bold", color="#55A868", fontsize=12)
ax.text(xs[0]+0.35, -2.4, "cloudy input\n+ neighbor days", ha="center", fontsize=10)
ax.text(xs[3]+0.35, -1.6, "bottleneck:\nlearns the pattern", ha="center", fontsize=10)
ax.text(xs[-1]+0.35, -2.4, "gap-free map", ha="center", fontsize=10)
ax.set_xlim(-0.6, 7.6); ax.set_ylim(-3.0, 2.9)
ax.set_title("Autoencoder: compress to a bottleneck, then reconstruct", weight="bold", fontsize=14)
plt.tight_layout(); plt.savefig("fig_autoencoder.png", dpi=150, bbox_inches="tight"); plt.show()

In [13]:
# --- fig 2: old-vs-new chunk coverage (needs IO.zarr) ---
import os, numpy as np, xarray as xr, matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy import ndimage
import cartopy.crs as ccrs, cartopy.feature as cfeature
import mindthegap as mtg
ORIGINAL = os.path.expanduser("~/shared/mind_the_chl_gap/IO.zarr")
th, tw = 40, 56
def masks(sub):
    s = mtg.crop_to_multiple(xr.open_zarr(ORIGINAL, chunks={}).sel(**sub), multiple=8)
    cv = "CHL" if "CHL" in s else [v for v in s.data_vars if "CHL" in v.upper()][0]
    ocean = ~s[cv].sel(time="2019").isnull().all("time").values
    coast = ocean & ndimage.binary_dilation(~ocean)
    ext = [float(s.lon.min()), float(s.lon.max()), float(s.lat.min()), float(s.lat.max())]
    return ocean, coast, ext
def positions(S, ocean, coast, rng):
    LAT, LON = ocean.shape; th_, tw_ = S["tile"] if S["tile"] else (LAT, LON)
    if S["sampling"] == "grid":
        oh, ow = S.get("overlap") or (0, 0); sh, sw = th_-oh, tw_-ow
        return [(i*sh, j*sw) for i in range((LAT-th_)//sh+1) for j in range((LON-tw_)//sw+1)]
    if S["sampling"] == "coast":
        n = S["n_per_day"]; mo = S.get("min_ocean", 0.5); nc = int(round(n*S.get("coast_frac", 1.0))); out = []
        ys, xs = np.where(coast)
        for i in rng.choice(len(ys), size=nc*8, replace=True):
            r, c = int(ys[i]), int(xs[i]); best, bf = None, -1
            for yy, xx in [(r-th_//2, c-tw_+1), (r-th_//2, c), (r, c-tw_//2)]:
                yy = int(np.clip(yy, 0, LAT-th_)); xx = int(np.clip(xx, 0, LON-tw_)); f = ocean[yy:yy+th_, xx:xx+tw_].mean()
                if f > bf: bf, best = f, (yy, xx)
            if bf >= mo: out.append(best)
            if len(out) >= nc: break
        while len(out) < n:
            yy = int(rng.integers(0, LAT-th_+1)); xx = int(rng.integers(0, LON-tw_+1))
            if ocean[yy:yy+th_, xx:xx+tw_].mean() >= mo: out.append((yy, xx))
        return out
    mo = S.get("min_ocean", 0.0); out = []
    while len(out) < S["n_per_day"]:
        yy = int(rng.integers(0, LAT-th_+1)); xx = int(rng.integers(0, LON-tw_+1))
        if mo <= 0 or ocean[yy:yy+th_, xx:xx+tw_].mean() >= mo: out.append((yy, xx))
    return out
def coverage(S, ocean, coast, days=150, seed=0):
    LAT, LON = ocean.shape; th_, tw_ = S["tile"] if S["tile"] else (LAT, LON)
    if S["tile"] is None: return np.ones((LAT, LON))
    cov = np.zeros((LAT, LON)); rng = np.random.default_rng(seed)
    for _ in range(1 if S["sampling"] == "grid" else days):
        for yy, xx in positions(S, ocean, coast, rng): cov[yy:yy+th_, xx:xx+tw_] += 1
    return cov
OLD_SUB = dict(lat=slice(31,5), lon=slice(42,80)); NEW_SUB = dict(lat=slice(25,5), lon=slice(50,78))
OLD_STR = {"whole":dict(tile=None,overlap=None,sampling="grid"),
           "nonoverlap":dict(tile=(40,56),overlap=(0,0),sampling="grid"),
           "overlap":dict(tile=(40,56),overlap=(20,28),sampling="grid"),
           "overlap_small":dict(tile=(40,56),overlap=(4,6),sampling="grid"),
           "random":dict(tile=(40,56),n_per_day=16,sampling="random",min_ocean=0.0)}
NEW_STR = {"nonoverlap":dict(tile=(40,56),overlap=(0,0),sampling="grid"),
           "overlap":dict(tile=(40,56),overlap=(20,28),sampling="grid"),
           "random":dict(tile=(40,56),n_per_day=16,sampling="random",min_ocean=0.0),
           "ocean_random":dict(tile=(40,56),n_per_day=16,sampling="random",min_ocean=0.5),
           "coast":dict(tile=(40,56),n_per_day=16,sampling="coast",coast_frac=0.5,min_ocean=0.5)}
oc_o, co_o, ext_o = masks(OLD_SUB); oc_n, co_n, ext_n = masks(NEW_SUB)
cmap3 = ListedColormap(["white", "lightgrey", "#55A868"])
fig, axes = plt.subplots(2, 5, figsize=(19, 8), subplot_kw={"projection": ccrs.PlateCarree()})
for row, (STR, oc, co, ext, tag) in enumerate([(OLD_STR, oc_o, co_o, ext_o, "OLD 104x152, 53% land"),
                                               (NEW_STR, oc_n, co_n, ext_n, "NEW 80x112, 27% land")]):
    for col, name in enumerate(STR):
        ax = axes[row, col]; cov = coverage(STR[name], oc, co)
        cat = np.zeros(oc.shape); cat[~oc] = 1; cat[(cov > 0) & oc] = 2
        ax.imshow(cat, extent=ext, origin="upper", transform=ccrs.PlateCarree(), cmap=cmap3, vmin=0, vmax=2)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
        ax.set_title(f"{name}\n{(cat==2).sum()/max(oc.sum(),1):.0%} ocean covered", size=9)
    for col in range(len(STR), 5): axes[row, col].axis("off")
    axes[row, 0].text(-0.16, 0.5, tag, transform=axes[row, 0].transAxes, rotation=90, va="center", ha="center", size=10)
plt.suptitle("Chunk coverage: OLD (top) vs NEW (bottom)   green = trained ocean, white = missed, grey = land", y=1.01)
plt.tight_layout(); plt.savefig("fig_coverage.png", dpi=150, bbox_inches="tight"); plt.show()

In [14]:
# --- fig 3: patch-strategy comparison, 5 seeds (numbers hardcoded from the run) ---
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
strat = ["overlap", "nonoverlap", "coast", "random", "ocean_random"]
mae   = [0.273, 0.279, 0.308, 0.311, 0.316]
std   = [0.013, 0.016, 0.028, 0.039, 0.032]
col   = ["#55A868", "#55A868", "#4C72B0", "#4C72B0", "#4C72B0"]
fig, ax = plt.subplots(figsize=(9, 5))
b = ax.bar(strat, mae, yerr=std, capsize=5, color=col, edgecolor="k")
for i, (bar, m) in enumerate(zip(b, mae)):
    ax.text(bar.get_x()+bar.get_width()/2, m+std[i]+0.006, f"{m:.3f}", ha="center", fontsize=10)
ax.set_ylabel("fake-cloud MAE (log Chl-a)", fontsize=12); ax.set_ylim(0, 0.38)
ax.set_title("Patch strategy, 5 seeds (lower = better): grids now beat crop-sampling", weight="bold", fontsize=12)
ax.legend(handles=[Patch(color="#55A868", label="grid tiles"), Patch(color="#4C72B0", label="crop sampling")], fontsize=11)
plt.tight_layout(); plt.savefig("fig_strategy_bars.png", dpi=150, bbox_inches="tight"); plt.show()

In [15]:
# --- fig 4: PACE cloud comparison (needs pace_arabsea_3day.zarr) ---
import numpy as np, xarray as xr, matplotlib.pyplot as plt
from scipy import ndimage
from scipy.ndimage import gaussian_filter
z = xr.open_zarr("pace_arabsea_3day.zarr")
chl = np.log(z["chlor_a"].where(z["chlor_a"] > 0).load().values).astype("float32")
gap = np.isnan(chl); land = gap.all(0); H, W = land.shape; oc = ~land
di = int(np.argsort((gap & ~land).sum((1, 2)))[len(chl)//2]); real = gap[di] & ~land
def keep(foot, span=0.7, asp=4.0):
    lab, n = ndimage.label(foot); k = np.zeros(n+1, bool)
    for i, sl in enumerate(ndimage.find_objects(lab), 1):
        if sl is None: continue
        h = sl[0].stop-sl[0].start; w = sl[1].stop-sl[1].start
        k[i] = not ((h >= span*H) or (w >= span*W) or max(h, w)/max(1, min(h, w)) > asp)
    return k[lab]
shape_kept = keep(real)
rng = np.random.default_rng(0); f = gaussian_filter(rng.standard_normal((H, W)), 6.0)
synth = (f > np.quantile(f, 0.6)) & ~land
cov = lambda m: (m & oc).sum()/oc.sum()
fig, ax = plt.subplots(1, 4, figsize=(18, 4.3))
for a, (arr, t, cmap) in zip(ax, [(chl[di], "observed chl", "viridis"),
                                  (real, f"real PACE gaps  {cov(real):.0%}", "gray_r"),
                                  (shape_kept, f"shape-filtered clouds  {cov(shape_kept):.0%}", "gray_r"),
                                  (synth, f"synthetic clouds  {cov(synth):.0%}", "gray_r")]):
    a.imshow(arr, cmap=cmap); a.set_title(t, size=11); a.axis("off")
plt.suptitle("PACE gaps are big orbital swaths; we train on smaller cloud-shaped holes", size=13)
plt.tight_layout(); plt.savefig("fig_pace_clouds.png", dpi=150, bbox_inches="tight"); plt.show()

In [16]:
# --- fig 5: temporal leak (numbers hardcoded from the AC experiment) ---
import numpy as np, matplotlib.pyplot as plt
off = [0.2959, 0.2674]; on = [0.2346, 0.2571]; persist = [0.2593, 0.2672]
x = np.arange(2); w = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x-w/2, off, w, label="U-Net, no prev/next", color="#8172B3", edgecolor="k")
ax.bar(x+w/2, on,  w, label="U-Net, with prev/next", color="#C44E52", edgecolor="k")
for i in range(2):
    ax.plot([x[i]-0.45, x[i]+0.45], [persist[i]]*2, "k--", lw=1.6)
    ax.text(x[i]+0.45, persist[i], " persistence", va="center", fontsize=9)
    ax.annotate(f"prev/next\n{'-21%' if i==0 else '-4%'}", (x[i], (off[i]+on[i])/2), ha="center", fontsize=10, weight="bold")
ax.set_xticks(x); ax.set_xticklabels(["independent clouds\n(what we had)", "correlated clouds\n(realistic)"], fontsize=11)
ax.set_ylabel("fake-cloud MAE (log Chl-a)", fontsize=12); ax.set_ylim(0, 0.36); ax.legend(fontsize=10)
ax.set_title("The temporal leak: prev/next 'helps' 21% under independent clouds,\nonly 4% once clouds persist realistically", weight="bold", fontsize=12)
plt.tight_layout(); plt.savefig("fig_temporal_leak.png", dpi=150, bbox_inches="tight"); plt.show()

In [17]:
# --- fig 6: 4-panel PACE prediction (needs pace_viz.npz; no GPU) ---
import numpy as np, xarray as xr, matplotlib.pyplot as plt
import cartopy.crs as ccrs, cartopy.feature as cfeature
d = np.load("pace_viz.npz", allow_pickle=True)
preds, truth, fake, land, times = d["preds"], d["truth"], d["fake"], d["land"], d["times"]
_z = xr.open_zarr("pace_arabsea_3day.zarr")
EXT = [float(_z.lon.min()), float(_z.lon.max()), float(_z.lat.min()), float(_z.lat.max())]
proj = ccrs.PlateCarree()
day = int(np.argmax((fake & np.isfinite(truth)).sum((1, 2))))
obs, pr, fk = truth[day], preds[day], fake[day]
flags = np.where(np.isfinite(obs) & ~fk, 2.0, np.where(fk, 1.0, 0.0))
vmin, vmax = np.nanpercentile(obs, [2, 98])
cm = plt.cm.viridis.copy(); cm.set_bad("lightgray")
panels = [(np.where(land, np.nan, obs),                "Observed L3 log Chl-a",            cm,       vmin, vmax, "log Chl-a"),
          (flags,                                      "flags: land/cloud=0, fake=1, obs=2","viridis", 0,    2,    "flag"),
          (np.where(land, np.nan, pr),                 "U-Net prediction",                 cm,       vmin, vmax, "log Chl-a"),
          (np.where(np.isfinite(obs), obs-pr, np.nan), "observed - predicted",             "RdBu",   -1,   1,    "difference")]
fig, axes = plt.subplots(2, 2, figsize=(13, 9), subplot_kw={"projection": proj})
for a, (arr, ttl, cmap, vmn, vmx, clab) in zip(axes.ravel(), panels):
    im = a.imshow(arr, extent=EXT, origin="upper", transform=proj, cmap=cmap, vmin=vmn, vmax=vmx, interpolation="nearest")
    a.add_feature(cfeature.COASTLINE, linewidth=0.4); a.set_title(ttl, size=11)
    fig.colorbar(im, ax=a, shrink=0.7, label=clab)
plt.suptitle(f"PACE gap-fill, {times[day]}", size=13)
plt.tight_layout(); plt.savefig("fig_pace_4panel.png", dpi=150, bbox_inches="tight"); plt.show()

In [18]:
# --- fig 7: per-day strategy error maps (needs seed-0 models; imports TF, holds the GPU) ---
import os, numpy as np, xarray as xr, pandas as pd, tensorflow as tf, matplotlib.pyplot as plt
import cartopy.crs as ccrs, cartopy.feature as cfeature
import mindthegap as mtg
for g in tf.config.list_physical_devices("GPU"): tf.config.experimental.set_memory_growth(g, True)
ORIGINAL = os.path.expanduser("~/shared/mind_the_chl_gap/IO.zarr")
SUBSET = dict(lat=slice(25, 5), lon=slice(50, 78))
train_year, train_range, TEST_YEAR, MODEL_DIR = 2015, 3, 2019, "models/spatial_strat"
def open_region(p):
    ds = mtg.crop_to_multiple(xr.open_zarr(p, chunks={}).sel(**SUBSET), multiple=8)
    return ds.sel(time=slice(f"{train_year}-01-01", f"{train_year+train_range+2}-01-01"))
_, STATS = mtg.build_standardized_lazy(open_region(ORIGINAL), [], train_year, train_range, standardize_chl=True)
ym, ys = STATS["CHL"]
ds_std, _ = mtg.build_standardized_lazy(open_region(ORIGINAL).sel(time=str(TEST_YEAR)), [],
                                        train_year, train_range, standardize_chl=True, stats=STATS)
ds_std = ds_std.load()
xv = [v for v in ds_std.data_vars if v != "CHL"]
ext = [float(ds_std.lon.min()), float(ds_std.lon.max()), float(ds_std.lat.min()), float(ds_std.lat.max())]
strategies = ["nonoverlap", "overlap", "random", "ocean_random", "coast"]
models = {n: tf.keras.models.load_model(f"{MODEL_DIR}/{n}_seed0.keras", compile=False)
          for n in strategies if os.path.exists(f"{MODEL_DIR}/{n}_seed0.keras")}
names = list(models); print("loaded:", names)
times = pd.to_datetime(ds_std.time.values)
obs_count = np.isfinite(ds_std["CHL"].values).reshape(len(times), -1).sum(1)
dates = [times[i] for i in np.sort(np.argsort(-obs_count)[:3])]
fig, axes = plt.subplots(len(dates), len(names), figsize=(2.6*len(names), 2.2*len(dates)),
                         subplot_kw={"projection": ccrs.PlateCarree()})
for r, dt in enumerate(dates):
    sub = ds_std.sel(time=dt)
    X = np.stack([np.nan_to_num(sub[v].values, nan=0.0) for v in xv], -1).astype("float32")
    truth = sub["CHL"].values * ys + ym
    for c, name in enumerate(names):
        pred = models[name].predict(X[None], verbose=0)[0, ..., 0] * ys + ym
        diff = np.where(np.isfinite(truth), truth - pred, np.nan)
        ax = axes[r, c]
        ax.imshow(diff, extent=ext, origin="upper", transform=ccrs.PlateCarree(), cmap="RdBu", vmin=-1, vmax=1)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.3)
        if r == 0: ax.set_title(name, size=10)
        if c == 0: ax.text(-0.12, 0.5, str(dt.date()), transform=ax.transAxes, rotation=90, va="center", ha="center", size=8)
plt.suptitle("Prediction error (observed - predicted): days x strategies   (red = under, blue = over)", y=1.0)
plt.tight_layout(); plt.savefig("fig_strategy_errormaps.png", dpi=150, bbox_inches="tight"); plt.show()

## Next steps (work in progress)

Done since the first pass: season-aware blocked split, shape/synthetic fake clouds, and the n_days
re-test (error flat across 1/2/3 days, variance grows with more days -> use n_days=1).

Open questions for the data scientists:

- **Cloud realism (the main caveat).** Our fake clouds cover ~40% of the ocean, but real PACE gaps are
  ~60%+ and swath-shaped, so the reported skill is optimistic. How should we calibrate synthetic
  coverage/shape to be honest without making the task impossible?
- **Coverage crossover.** The U-Net beats persistence at low coverage but loses around ~50-60% (spatial
  fill needs surrounding pixels; when most are missing, yesterday's value wins). Worth reporting
  MAE-vs-coverage rather than a single number.
- **Which split to report.** Contiguous future-holdout (conservative, operational fill) vs season-blocked
  (interspersed, optimistic, archive fill) - a use-case call for the mentors.
- Later: scale past one region (tiling + `tiled_predict` stitching + positional channels).

See `book/PACE_global_scaling_design.md` for the broader scaling plan and open questions.